# rounds

> Capture a session from any host, review it, accept it, and start the next session from it.

In [ ]:
#| default_exp rounds

In [ ]:
#| export
from pathlib import Path
import json, shlex, subprocess, sys

from aidialog.dialog import Dialog, snote, sprompt
from aidialog.hist import chat2dlg, dlg2chat
from aidialog.ipynb import read_ipynb, write_ipynb
from aidialog.msg_parts import Msg, Text, ToolUse, ToolResult
from fastcore.script import call_parse
from llmsurgery import ant, oai
from llmsurgery.sess import path_dlg, sess_dlg
from urai import ToolCall, mk_tool_res_msg

from drona.core import RAMABANA_HISTORY, assess_history, read_history

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
import tempfile

A round is one conversation, reviewed by a person, that later conversations start from. It lives as
an Aidialog notebook, which is the interchange format: a Ramabana, Claude Code, or Codex session
becomes one, a person edits it in Leela, and it compiles back out to whichever host runs next.

The notebook moves through three states, kept under a `drona` key in its metadata. `capture` writes
it as `review`, `accept` records a named reviewer and marks it `accepted`, and every compiler
refuses a notebook that has not reached that second state. Nothing leaves Drona unreviewed, on any
host, because the review is where private material and the routes not worth imitating come out.

In [ ]:
#| export
REVIEW_KEY = 'drona'
HOSTS = ('ramabana', 'claude', 'codex')
ROUND_REVISION = 2
BOOTSTRAP_DETAIL = 600
REVIEW_NOTE = ('# Drona review\n\n'
               'Edit this dialog in Leela. Delete the detours and anything private, keep the route a '
               'later model should imitate, then run `drona-accept` on it.')

def drona_version():
    "The package version and round revision stamped into every accepted round."
    from drona import __version__
    return f'{__version__}:{ROUND_REVISION}'

def _clip(text, n):
    s = str(text or '')
    return s if len(s) <= n else s[:n] + f'\n…[{len(s)-n} more chars]'

def _read(source):
    dlg = read_ipynb(source)
    if not dlg: raise ValueError(f'could not read dialog {source}')
    return dlg

def _prompts(dlg): return [m for m in dlg if m.msg_type == sprompt and not m.skipped]

## Capture a session

Capture reads an archive after the conversation has ended. It never watches a live process.

Ramabana keeps its own turn archive, so its capture goes through `read_history` and scores the route
on the way past. Claude and Codex are read by llmsurgery. That reader resolves a session id without
being told which host it belongs to, so Drona asks for the host anyway and refuses a session that
turns out to be the other one, rather than filing a Codex thread under `claude`.

In [ ]:
#| export
LS_HOSTS = {'claude': 'ant', 'codex': 'oai'}

def turn_msgs(turn):
    "One Ramabana archive turn as typed Aidialog messages."
    msgs = [Msg('user', [Text(str(turn.get('prompt') or ''))])]
    for i, a in enumerate(turn.get('activity') or ()):
        cid, tool = a.get('action_id') or a.get('id') or f'call_{i}', a.get('tool', '')
        args, detail = a.get('args') or {}, str(a.get('detail') or '')
        msgs += [Msg('assistant', [ToolUse(id=cid, name=tool, arguments=args)]),
                 Msg('tool', [ToolResult(id=cid, name=tool, arguments=args, text=detail)])]
    if reply := str(turn.get('reply') or ''): msgs.append(Msg('assistant', [Text(reply)]))
    return msgs

def _latest_path(host, cwd, codex_home):
    "The newest session file on `host`."
    if host == 'codex': return oai.project_thread(cwd or '.', codex_home or oai.CODEX_HOME)[1]
    paths = sorted(ant.sess_dir(cwd).glob('*.jsonl'), key=lambda p: p.stat().st_mtime)
    if not paths: raise ValueError(f'no Claude session under {ant.sess_dir(cwd)}')
    return paths[-1]

def host_dlg(
    host,             # `claude` or `codex`
    session='latest', # session id, id prefix, or `latest`
    cwd=None,         # project directory
    codex_home=None,  # Codex home
    name=None,        # dialog name
):
    "One Claude or Codex session as a dialog, checked against the host that was asked for."
    if session == 'latest':
        path = _latest_path(host, cwd, codex_home)
        dlg = path_dlg(LS_HOSTS[host], path, name=name or path.stem, mx=None)
    else: dlg = sess_dlg(session, cwd=cwd, codex_home=codex_home, name=name, mx=None)
    got = (dlg.meta.get('llmsurgery') or {}).get('host')
    if got != LS_HOSTS[host]: raise ValueError(f'session {session!r} belongs to {got!r}, not {host!r}')
    return dlg

def capture(
    output,                   # review notebook path
    host='ramabana',          # `ramabana`, `claude`, or `codex`
    session='latest',         # session id, id prefix, or `latest`
    cwd=None,                 # project directory, for Claude and Codex
    history=RAMABANA_HISTORY, # Ramabana history path
    codex_home=None,          # Codex home
    name=None,                # round name; the output stem when omitted
):
    "Capture one host session as a Drona review notebook."
    if host not in HOSTS: raise ValueError(f'host must be one of {HOSTS}')
    output, name = Path(output), name or Path(output).stem
    if host == 'ramabana':
        turns = read_history(history, session)
        dlg = chat2dlg([m for t in turns for m in turn_msgs(t)], name, mx=None)
        found = {'session': turns[0].get('session'), **assess_history(turns).dict()}
    else:
        dlg = host_dlg(host, session, cwd, codex_home, name)
        found = dict(dlg.meta.get('llmsurgery') or {})
    review = {'version': drona_version(), 'status': 'review', **found, 'host': host}
    dlg.meta[REVIEW_KEY] = review
    dlg.mk_message(REVIEW_NOTE, idx=0, msg_type=snote, skipped=1, meta={REVIEW_KEY: review})
    output.parent.mkdir(parents=True, exist_ok=True)
    write_ipynb(dlg, output)
    return output

A captured Ramabana round carries its score, so a reviewer knows before reading which routes the
archive already thinks are poor.

In [ ]:
tmp = Path(tempfile.mkdtemp())
archive = tmp/'agent-history.jsonl'
turn = {'session': 'sess-aaa', 'state': 'complete',
        'prompt': 'Use fossick to research the AnswerDotAI llmdojo github repository',
        'reply': 'FOSSICK read the repository directly.',
        'activity': [{'action_id': 'a0', 'tool': 'run_shell', 'ok': True,
                      'args': {'command': 'fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo'},
                      'detail': '# llmdojo\nLLM coding agents imitate what their context shows.'}]}
archive.write_text(json.dumps(turn) + '\n')

review = capture(tmp/'round.ipynb', history=archive)
captured = read_ipynb(review)
test_eq(captured.meta[REVIEW_KEY]['status'], 'review')
test_eq(captured.meta[REVIEW_KEY]['host'], 'ramabana')
test_eq(captured.meta[REVIEW_KEY]['score'], 100)
test_fail(lambda: capture(tmp/'x.ipynb', host='other'), contains='host must be one of')

## Accept a round

Acceptance is explicit and attributed. It checks that the reviewed dialog still round trips to a
history that opens with a user turn, records who accepted it, and writes the compiled history beside
the notebook as derived data.

In [ ]:
#| export
def _part_dict(part):
    kind = getattr(part.type, 'value', None) or str(part.type)
    return {'type': kind, **{k: v for k, v in vars(part).items()
                             if k != 'raw' and v not in (None, False, {}, [])}}

def _msg_dict(msg): return {'role': msg.role, 'content': [_part_dict(p) for p in msg.content]}

def accepted(source):
    "The reviewed prompts of an accepted round, refusing a round nobody has accepted."
    dlg = _read(source)
    meta = dlg.meta.get(REVIEW_KEY) or {}
    if meta.get('status') != 'accepted':
        raise ValueError(f'{source} is not accepted; run drona-accept on it first')
    return Dialog(_prompts(dlg), name=dlg.name, meta=dlg.meta)

def compiled_history(source):
    "Canonical Aidialog history from an accepted round."
    return dlg2chat(accepted(source), plain=True)

def accept(
    source,      # reviewed Drona notebook
    reviewer,    # the person accepting it
    output=None, # compiled JSON path; `<source>.json` when omitted
):
    "Accept a reviewed round and write its canonical history."
    if not str(reviewer).strip(): raise ValueError('a round needs a named reviewer')
    dlg = _read(source)
    history = dlg2chat(Dialog(_prompts(dlg), name=dlg.name), plain=True)
    if not history or history[0].role != 'user': raise ValueError('a round must open with a user turn')
    meta = {**(dlg.meta.get(REVIEW_KEY) or {}), 'status': 'accepted',
            'reviewer': str(reviewer), 'accepted_version': drona_version()}
    dlg.meta[REVIEW_KEY] = meta
    dlg.save(source)
    output = Path(output) if output else Path(source).with_suffix('.json')
    output.write_text(json.dumps({'meta': meta, 'history': [_msg_dict(m) for m in history]}, indent=2))
    return output

In [ ]:
test_fail(lambda: compiled_history(review), contains='is not accepted')
test_fail(lambda: accept(review, '  '), contains='named reviewer')

compiled = json.loads(accept(review, 'Karthik').read_text())
test_eq(compiled['meta']['status'], 'accepted')
test_eq(compiled['meta']['reviewer'], 'Karthik')
test_eq(compiled['history'][0]['role'], 'user')
test_eq([m.role for m in compiled_history(review)], ['user', 'assistant', 'tool', 'assistant'])

## Compile a round for the next session

An accepted round leaves as one of three shapes. Urai history feeds a chat directly. A bootstrap
prompt carries the round to a host whose command line takes no prepared history, which is what
Ramabana needs. A host session file is what Claude Code and Codex read.

Tool results in a bootstrap prompt are clipped to the same 600 characters Ramabana itself keeps
when it replays a saved turn, so a long round still fits on a command line.

In [ ]:
#| export
def warm_start(source):
    "An accepted round as Urai history, for `messages=` on any Urai or Rishi chat."
    out = []
    for m in compiled_history(source):
        if m.role == 'user':
            out.append({'role': 'user', 'content': m.text})
        elif m.role == 'assistant':
            text = ''.join(p.text for p in m.content if isinstance(p, Text) and p.text)
            calls = [ToolCall(p.name, p.arguments, id=p.id) for p in m.content if isinstance(p, ToolUse)]
            out.append({'role': 'assistant', 'content': text, **({'tool_calls': calls} if calls else {})})
        elif m.role == 'tool':
            out += [mk_tool_res_msg(ToolCall(p.name, p.arguments, id=p.id), p.text)
                    for p in m.content if isinstance(p, ToolResult)]
    return out

def prepare_chat(
    chat,   # an empty Urai or Rishi chat
    source, # accepted round notebook
):
    "Prepend an accepted round to an empty Urai-compatible chat."
    if chat.hist: raise ValueError('Drona prepares an empty chat only')
    chat.hist = chat.fmt2hist(warm_start(source))
    if hasattr(chat, '_recreate_conv'): chat._recreate_conv()
    return chat

def bootstrap_prompt(
    source,                  # accepted round notebook
    detail=BOOTSTRAP_DETAIL, # characters of each tool result to keep
):
    "An accepted round as one prompt, for a host whose command line takes no prepared history."
    rows = ['The reviewed Drona round below is the tool route to follow.']
    for m in compiled_history(source):
        if m.role == 'user':
            rows.append(f'User: {m.text}')
            continue
        for p in m.content:
            if isinstance(p, ToolUse):
                rows.append(f'Assistant tool: {p.name}({json.dumps(p.arguments, sort_keys=True, default=str)})')
            elif isinstance(p, ToolResult): rows.append(f'Tool result: {_clip(p.text, detail)}')
            elif isinstance(p, Text) and p.text: rows.append(f'Assistant: {p.text}')
    rows.append('Reply with exactly: DRONA_READY')
    return '\n\n'.join(rows)

def export_round(
    source,      # accepted round notebook
    host,        # `ramabana`, `claude`, or `codex`
    output=None, # file to write; Claude writes into its own session directory instead
    cwd=None,    # project directory, for Claude
):
    "Compile an accepted round for one host."
    if host not in HOSTS: raise ValueError(f'host must be one of {HOSTS}')
    if host == 'claude': return ant.dlg2sess(accepted(source), cwd=cwd)
    out = bootstrap_prompt(source) if host == 'ramabana' else list(oai.dlg2items(accepted(source)))
    if output: Path(output).write_text(out if isinstance(out, str) else json.dumps(out, indent=2))
    return out

Urai history keeps the tool call and its result paired by the same id.

In [ ]:
hist = warm_start(review)
test_eq([m['role'] for m in hist], ['user', 'assistant', 'tool', 'assistant'])
test_eq(hist[1]['tool_calls'][0].name, 'run_shell')
test_eq(hist[2]['tool_call_id'], hist[1]['tool_calls'][0]['id'])

`prepare_chat` puts the same history on an empty chat, and refuses a chat that already has one. A chat only has to carry `hist` and `fmt2hist` for this, so no model is loaded.

In [ ]:
class FakeChat:
    def __init__(self): self.hist = []
    def fmt2hist(self, msgs): return list(msgs)

chat = prepare_chat(FakeChat(), review)
test_eq([m['role'] for m in chat.hist], ['user', 'assistant', 'tool', 'assistant'])
test_fail(lambda: prepare_chat(chat, review), contains='empty chat only')

Every compiler goes through `accepted`, so an unreviewed notebook cannot reach any host.

In [ ]:
unreviewed = capture(tmp/'unreviewed.ipynb', history=archive)
for host in HOSTS: test_fail(lambda: export_round(unreviewed, host), contains='is not accepted')

prompt = export_round(review, 'ramabana', output=tmp/'round.txt')
assert prompt.endswith('Reply with exactly: DRONA_READY')
assert 'fossick read-gh-repo' in prompt
test_eq((tmp/'round.txt').read_text(), prompt)

A long tool result is clipped rather than pasted whole onto a command line.

In [ ]:
big = dict(turn, activity=[dict(turn['activity'][0], detail='x'*5000)])
(tmp/'big.jsonl').write_text(json.dumps(big) + '\n')
accept(capture(tmp/'big.ipynb', history=tmp/'big.jsonl'), 'Karthik')
assert len(bootstrap_prompt(tmp/'big.ipynb')) < 2000
assert 'more chars]' in bootstrap_prompt(tmp/'big.ipynb')

Codex items keep one call id across the function call and its output.

In [ ]:
items = export_round(review, 'codex', output=tmp/'items.json')
call = next(i for i in items if i['type'] == 'function_call')
result = next(i for i in items if i['type'] == 'function_call_output')
test_eq(call['call_id'], result['call_id'])
test_eq(call['name'], 'run_shell')

## Start the next session

Ramabana takes no prepared history on its command line, so `start_round` sends the round as one
bootstrap turn and then resumes the session that turn created. Ramabana exits non-zero whenever it
finished with anything to report, which is routine, so the bootstrap's exit code is surfaced and
the resume goes ahead regardless.

In [ ]:
#| export
def start_commands(
    source,     # accepted round notebook
    root='.',   # Ramabana root folders, comma separated
    model=None, # optional model name
    cfg=None,   # Ramabana config dir, when it is not the default
):
    "The Ramabana bootstrap and resume commands for an accepted round."
    base = ['ramabana', '--root', str(root)]
    if cfg: base += ['--cfg', str(cfg)]
    if model: base += ['--model', model]
    return base + [bootstrap_prompt(source)], base + ['--resume', 'latest']

def start_round(
    source,      # accepted round notebook
    root='.',    # Ramabana root folders, comma separated
    model=None,  # optional model name
    cfg=None,    # Ramabana config dir, when it is not the default
    launch=False, # run the commands, rather than return them
):
    "Prepare or launch Ramabana with an accepted Drona round."
    first, resume = start_commands(source, root, model, cfg)
    if not launch: return {'bootstrap': first, 'resume': resume}
    if rc := subprocess.run(first).returncode: print(f'bootstrap exited {rc}', file=sys.stderr)
    return subprocess.run(resume).returncode

In [ ]:
first, resume = start_commands(review, root='/tmp/project', model='sonnet', cfg='/tmp/cfg')
test_eq(first[:8], ['ramabana', '--root', '/tmp/project', '--cfg', '/tmp/cfg', '--model', 'sonnet',
                    bootstrap_prompt(review)])
test_eq(resume[-2:], ['--resume', 'latest'])
test_eq(start_round(review, root='/tmp/project')['resume'][:3], ['ramabana', '--root', '/tmp/project'])

## Command line

Five commands follow the round's life: assess an archive, capture a session, accept it, compile it
for a host, and start the next session from it.

In [ ]:
#| export
@call_parse
def capture_cli(source: str, host: str='ramabana', session: str='latest', cwd: str=None,
                history: str=str(RAMABANA_HISTORY), codex_home: str=None, name: str=None):
    "Capture a host session as a Drona review notebook."
    print(capture(source, host, session, cwd, history, codex_home, name))

@call_parse
def accept_cli(source: str, reviewer: str, output: str=None):
    "Accept a reviewed round and compile its history."
    print(accept(source, reviewer, output))

@call_parse
def export_cli(source: str, host: str, output: str=None, cwd: str=None):
    "Compile an accepted round for one host."
    out = export_round(source, host, output, cwd)
    if output and host != 'claude': print(output)
    else: print(out if isinstance(out, str) else json.dumps(out, indent=2))

@call_parse
def start_cli(source: str, root: str='.', model: str=None, cfg: str=None, launch: bool=False):
    "Prepare or launch Ramabana with an accepted round."
    if isinstance(out := start_round(source, root, model, cfg, launch), dict):
        for name, cmd in out.items(): print(f'{name}: {shlex.join(cmd)}')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()